# Donor Models — Training

Trains two models from `warehouse.fact_donors_ml`:

1. **Churn classifier** — Will a donor stop donating? (`churned`: 0 / 1)
2. **Donation type classifier** — What type will their next donation be? (`Monetary / In-Kind / Time / Skills`)

Outputs: `donor_churn_model.sav`, `donor_type_model.sav` + metadata + metrics JSON files.

In [ ]:
import sys
sys.path.insert(0, '../jobs')

import json
from datetime import datetime, timezone

import joblib
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from config import (
    ARTIFACTS_DIR,
    DONOR_CHURN_METADATA_PATH, DONOR_CHURN_METRICS_PATH, DONOR_CHURN_MODEL_PATH,
    DONOR_TYPE_METADATA_PATH, DONOR_TYPE_METRICS_PATH, DONOR_TYPE_MODEL_PATH,
    WAREHOUSE_SCHEMA,
)
from utils_db import get_engine

print('Imports loaded.')

## Load data

In [ ]:
engine = get_engine(WAREHOUSE_SCHEMA)
df = pd.read_sql('SELECT * FROM fact_donors_ml', engine)
print(f'Loaded {len(df)} rows')
df.head()

## Feature columns

In [ ]:
FEATURE_COLS = [
    'supporter_type_encoded', 'region_encoded', 'channel_encoded', 'relationship_encoded',
    'total_donations', 'total_value', 'avg_value', 'is_recurring',
    'days_since_last', 'first_donation_days_ago', 'num_campaigns',
]

print(f'Churn distribution:\n{df["churned"].value_counts().to_string()}')
print(f'\nDonation type distribution:\n{df["last_donation_type"].value_counts().to_string()}')

## Model 1 — Churn classifier

In [ ]:
available = [c for c in FEATURE_COLS if c in df.columns]
X_churn = df[available]
y_churn = df['churned'].astype(int)

min_class = y_churn.value_counts().min()
stratify = y_churn if min_class >= 2 else None
X_train, X_test, y_train, y_test = train_test_split(
    X_churn, y_churn, test_size=0.25, random_state=42, stratify=stratify
)

churn_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')),
])
churn_pipeline.fit(X_train, y_train)

y_pred   = churn_pipeline.predict(X_test)
accuracy = float(accuracy_score(y_test, y_pred))
f1       = float(f1_score(y_test, y_pred, average='weighted', zero_division=0))
report   = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

print(f'Churn — Accuracy: {accuracy:.3f} | Weighted F1: {f1:.3f}')
print(classification_report(y_test, y_pred, zero_division=0))

## Save churn model

In [ ]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(churn_pipeline, str(DONOR_CHURN_MODEL_PATH))

with open(DONOR_CHURN_METADATA_PATH, 'w') as f:
    json.dump({
        'model_name': 'donor_churn_classifier', 'model_version': '1.0.0',
        'trained_at_utc': datetime.now(timezone.utc).isoformat(),
        'warehouse_table': 'fact_donors_ml',
        'num_training_rows': int(len(X_train)), 'num_test_rows': int(len(X_test)),
        'label_col': 'churned', 'feature_cols': available,
        'classes': [int(c) for c in churn_pipeline.classes_],
    }, f, indent=2)
with open(DONOR_CHURN_METRICS_PATH, 'w') as f:
    json.dump({'accuracy': accuracy, 'weighted_f1': f1, 'classification_report': report}, f, indent=2)

print(f'Saved: {DONOR_CHURN_MODEL_PATH}')

## Model 2 — Donation type classifier

In [ ]:
df_type = df.dropna(subset=['last_donation_type'])
available_t = [c for c in FEATURE_COLS if c in df_type.columns]
X_type = df_type[available_t]
y_type = df_type['last_donation_type']

min_class = y_type.value_counts().min()
stratify = y_type if min_class >= 2 else None
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_type, y_type, test_size=0.25, random_state=42, stratify=stratify
)

type_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('clf',     GradientBoostingClassifier(n_estimators=100, random_state=42)),
])
type_pipeline.fit(X_train_t, y_train_t)

y_pred_t   = type_pipeline.predict(X_test_t)
accuracy_t = float(accuracy_score(y_test_t, y_pred_t))
f1_t       = float(f1_score(y_test_t, y_pred_t, average='weighted', zero_division=0))
report_t   = classification_report(y_test_t, y_pred_t, output_dict=True, zero_division=0)

print(f'Type — Accuracy: {accuracy_t:.3f} | Weighted F1: {f1_t:.3f}')
print(classification_report(y_test_t, y_pred_t, zero_division=0))

## Save donation type model

In [ ]:
joblib.dump(type_pipeline, str(DONOR_TYPE_MODEL_PATH))

with open(DONOR_TYPE_METADATA_PATH, 'w') as f:
    json.dump({
        'model_name': 'donor_type_classifier', 'model_version': '1.0.0',
        'trained_at_utc': datetime.now(timezone.utc).isoformat(),
        'warehouse_table': 'fact_donors_ml',
        'num_training_rows': int(len(X_train_t)), 'num_test_rows': int(len(X_test_t)),
        'label_col': 'last_donation_type', 'feature_cols': available_t,
        'classes': list(type_pipeline.classes_),
    }, f, indent=2)
with open(DONOR_TYPE_METRICS_PATH, 'w') as f:
    json.dump({'accuracy': accuracy_t, 'weighted_f1': f1_t, 'classification_report': report_t}, f, indent=2)

print(f'Saved: {DONOR_TYPE_MODEL_PATH}')